# 🤖 CodeBERT Fine-Tuning — PR Risk Scorer

**Smit Final Project — JIT Defect Prediction**

This notebook fine-tunes `microsoft/codebert-base` on the curated ApacheJIT subset to classify bug-inducing commits.  
All checkpoints are saved to **Google Drive** so Colab disconnects are safe.

---
**Runtime required:** GPU (Runtime → Change runtime type → T4 GPU)  
**Estimated time:** ~15-25 min per epoch on T4


## Step 1: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ✅ CHANGE THIS to your preferred Drive folder
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/smit_final_project'
CHECKPOINT_DIR    = f'{DRIVE_PROJECT_DIR}/checkpoints/codebert'
DATA_DIR          = f'{DRIVE_PROJECT_DIR}/data/processed'
TRAIN_CSV         = f'{DATA_DIR}/train_features.csv'
TEST_CSV          = f'{DATA_DIR}/test_features.csv'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print('✅ Drive mounted. Paths ready.')
print(f'   CHECKPOINT_DIR = {CHECKPOINT_DIR}')
print(f'   TRAIN_CSV      = {TRAIN_CSV}')

## Step 2: Upload Dataset CSV files

Upload `train_features.csv` and `test_features.csv` from your local `data/processed/` folder to Drive.

> **Option A**: Upload via the Files panel on the left and move to Drive  
> **Option B**: Run the cell below to upload directly

In [ ]:
# Run this ONLY if you haven't already put the CSVs in Drive
from google.colab import files
import shutil

print('Upload train_features.csv and test_features.csv when prompted...')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = os.path.join(DATA_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  ✅ Saved {fname} → {dest}')

## Step 3: Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.40.2 torch accelerate scikit-learn pandas

## Step 4: Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU (T4)')

## Step 5: Clone / Copy Project Code (features/schema.py)

In [ ]:
# Clone your GitHub repo so we have CommitFeatures schema available
# ✅ CHANGE THIS to your actual repo URL
REPO_URL = 'https://github.com/funterpie/smit-final-project.git'

!git clone {REPO_URL} /content/pr_risk_scorer
import sys
sys.path.insert(0, '/content/pr_risk_scorer')
print('✅ Repo cloned and added to Python path')

## Step 6: Define Dataset & Training Config

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
MODEL_NAME  = 'microsoft/codebert-base'
NUM_EPOCHS  = 5
BATCH_SIZE  = 16
LR          = 2e-5
MAX_LEN     = 128
WARMUP_FRAC = 0.1
SEED        = 42
# ─────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Step 7: Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

FEATURE_COLS = [
    'lines_added', 'lines_removed', 'files_touched', 'dispersion',
    'author_bug_rate', 'hours_since_last_commit', 'tests_added',
    'touches_sensitive_module',
]

train_df = pd.read_csv(TRAIN_CSV).fillna(0)
test_df  = pd.read_csv(TEST_CSV).fillna(0)

print(f'Train: {len(train_df):,} rows | Positive rate: {train_df["buggy"].mean():.3f}')
print(f'Test : {len(test_df):,} rows  | Positive rate: {test_df["buggy"].mean():.3f}')
train_df[FEATURE_COLS].describe()

## Step 8: Build Dataset & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def features_to_text(row: pd.Series) -> str:
    """Convert feature row to a short structured text (no raw diff — by design)."""
    return (
        f"lines_added {int(row['lines_added'])} "
        f"lines_removed {int(row['lines_removed'])} "
        f"files_touched {int(row['files_touched'])} "
        f"dispersion {float(row['dispersion']):.3f} "
        f"author_bug_rate {float(row['author_bug_rate']):.3f} "
        f"hours_since_last_commit {float(row['hours_since_last_commit']):.1f} "
        f"tests_added {int(row['tests_added'])} "
        f"touches_sensitive_module {int(row['touches_sensitive_module'])}"
    )

class CommitDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.texts  = [features_to_text(row) for _, row in df.iterrows()]
        self.labels = df['buggy'].astype(int).tolist()
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = CommitDataset(train_df, tokenizer)
test_ds  = CommitDataset(test_df,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'✅ Dataset ready. Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## Step 9: Load Model (resume from checkpoint if exists)

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BEST_MODEL_DIR = os.path.join(CHECKPOINT_DIR, 'best_model')

if os.path.exists(BEST_MODEL_DIR):
    print(f'🔄 Resuming from checkpoint: {BEST_MODEL_DIR}')
    model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR)
else:
    print(f'🆕 Starting fresh from {MODEL_NAME}')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

model = model.to(DEVICE)
print(f'✅ Model loaded on {DEVICE}')

## Step 10: Training Loop (with checkpoint save every epoch)

In [ ]:
import torch.nn.functional as F

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

def evaluate(loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
            ).logits
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
    return np.array(all_probs), np.array(all_labels)

best_aucpr = 0.0
history    = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['labels'].to(DEVICE),
        )
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{len(train_loader)} | loss={outputs.loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)

    # ── Evaluate ────────────────────────────────────────────────────
    probs, labels = evaluate(test_loader)
    aucpr  = average_precision_score(labels, probs)
    aucroc = roc_auc_score(labels, probs)

    row = {'epoch': epoch, 'loss': avg_loss, 'auc_pr': aucpr, 'auc_roc': aucroc}
    history.append(row)
    print(f'\n📊 Epoch {epoch}/{NUM_EPOCHS} | loss={avg_loss:.4f} | AUC-PR={aucpr:.4f} | AUC-ROC={aucroc:.4f}')

    # ── Save checkpoint every epoch (Colab disconnect safety) ────────
    epoch_dir = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch}')
    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    print(f'  💾 Checkpoint saved → {epoch_dir}')

    # ── Save best model ─────────────────────────────────────────────
    if aucpr > best_aucpr:
        best_aucpr = aucpr
        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        print(f'  🏆 New best AUC-PR={best_aucpr:.4f} → saved best_model')

print('\n✅ Training complete!')
print(f'   Best AUC-PR: {best_aucpr:.4f}')
print(f'   Best model:  {BEST_MODEL_DIR}')

## Step 11: Plot Training History

In [ ]:
import matplotlib.pyplot as plt

epochs    = [h['epoch']   for h in history]
losses    = [h['loss']    for h in history]
aucprs    = [h['auc_pr']  for h in history]
aucrocs   = [h['auc_roc'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, losses,  'b-o'); axes[0].set_title('Training Loss');  axes[0].set_xlabel('Epoch')
axes[1].plot(epochs, aucprs,  'g-o'); axes[1].set_title('AUC-PR');         axes[1].set_xlabel('Epoch')
axes[2].plot(epochs, aucrocs, 'r-o'); axes[2].set_title('AUC-ROC');        axes[2].set_xlabel('Epoch')

plt.tight_layout()
plot_path = os.path.join(CHECKPOINT_DIR, 'training_history.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved → {plot_path}')

print('\n── Final Results ──')
for h in history:
    print(f"  Epoch {h['epoch']}: loss={h['loss']:.4f}  AUC-PR={h['auc_pr']:.4f}  AUC-ROC={h['auc_roc']:.4f}")

## Step 12: Download Best Model for Local Inference

After training, download the `best_model/` folder and place it at:  
`models/artefacts/codebert/best_model/`

That path is what `models/codebert.py → predict_codebert()` reads from.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/codebert_best_model.zip'
shutil.make_archive('/content/codebert_best_model', 'zip', BEST_MODEL_DIR)
files.download(zip_path)
print(f'✅ Download triggered: {zip_path}')
print('Extract and copy to: models/artefacts/codebert/best_model/')

## Step 13: Quick Sanity Check — Single Prediction

In [ ]:
model.eval()

# Simulate a high-risk commit
sample_text = (
    'lines_added 450 lines_removed 120 files_touched 12 dispersion 0.850 '
    'author_bug_rate 0.420 hours_since_last_commit 1.5 tests_added 0 touches_sensitive_module 1'
)

enc = tokenizer(sample_text, return_tensors='pt', max_length=MAX_LEN,
                truncation=True, padding='max_length').to(DEVICE)

with torch.no_grad():
    logits = model(**enc).logits
    prob   = float(F.softmax(logits, dim=-1)[0, 1])

print(f'Bug-inducing probability: {prob:.4f} ({"HIGH RISK" if prob > 0.5 else "LOW RISK"})')